# Centipede: test suite on a Kaggle GPU

This notebook runs the project's tests on a Kaggle T4, where MuJoCo Warp can use
its default Newton solver. It copies no project code: it clones the public
repositories of Centipede and of its two sibling packages, RL_lib and
MujocoReplay, installs them as in the README, and runs `pytest`.

Before running, in the notebook's side panel:

- **Accelerator:** GPU T4. The P100 cannot compile the Newton solver.
- **Internet:** on (needed for `git clone` and `pip`).

Then choose **Run all** and keep the output of every cell.

In [ ]:
# The machine: GPU model, driver, and Python version.
!nvidia-smi
!python --version

In [ ]:
# Fetch the project and the two sibling packages it installs. Change BRANCH
# to profile another branch of Centipede.
OWNER = "https://github.com/DrippingSoup22"
BRANCH = "main"

!rm -rf /kaggle/working/centipede /kaggle/working/RL_lib /kaggle/working/MujocoReplay
!git clone --depth 1 --branch {BRANCH} {OWNER}/centipede-project.git /kaggle/working/centipede
!git clone --depth 1 {OWNER}/RL_lib.git /kaggle/working/RL_lib
!git clone --depth 1 {OWNER}/MujocoReplay.git /kaggle/working/MujocoReplay
%cd /kaggle/working/centipede
!git log -1 --format="Centipede commit %h: %s"
!git -C /kaggle/working/RL_lib log -1 --format="RL_lib commit %h: %s"
!git -C /kaggle/working/MujocoReplay log -1 --format="MujocoReplay commit %h: %s"

In [ ]:
# Install RL_lib (the learning algorithms) and MujocoReplay (which writes the
# replay recordings), then the package with the GPU backend and the test tools. Kaggle
# already provides PyTorch with CUDA, so it is not downloaded again.
!pip install --quiet -e /kaggle/working/RL_lib -e /kaggle/working/MujocoReplay
!pip install --quiet -e ".[gpu,dev]"

In [ ]:
# The versions in use, and the GPU as PyTorch and Warp see it.
import mujoco
import mujoco_warp
import torch
import warp

print("MuJoCo", mujoco.__version__, "| MuJoCo Warp", mujoco_warp.__version__)
print("Warp", warp.__version__, "| PyTorch", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(),
    "| compute capability",
    torch.cuda.get_device_capability(),
    "|",
    torch.cuda.device_count(),
    "visible",
)
assert torch.cuda.get_device_capability() >= (7, 0), "Choose the T4 accelerator"

In [ ]:
# The full test suite. On this GPU the GPU tests use the Newton solver, and the
# agents' tests also run on CUDA.
!python -m pytest -q --durations=10